##Ingest Dimension Data into Bronze Layer

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

##Brands

In [0]:
#define schema for the data file
#this is for the brands data file
brand_schema = StructType([
    StructField('brand_code', StringType(), False),
    StructField('brand_name', StringType(), True),
    StructField('category_code',StringType(),True)
])

In [0]:
raw_data_path ="/Volumes/ecommerce/source_data/raw/brands/*.csv"
df = spark.read.option('header','true').option('delimiter',",").schema(brand_schema).csv(raw_data_path)

#adding metadata columns
df = df.withColumn("_source_file", F.col("_metadata.file_path"))\
    .withColumn("ingested_at",F.current_timestamp())

display(df.limit(5))

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_brands")

##Category


In [0]:
#define schema for the data file
#this is for the brands data file
category_schema = StructType([
    StructField('category_code', StringType(), False), #False because
    StructField('category_name', StringType(), True)
])

In [0]:
#load data using schema defined

raw_data_path ="/Volumes/ecommerce/source_data/raw/category/*.csv"
df_cat = spark.read.option("header","true").option("delimiter",",").schema(category_schema).csv(raw_data_path)

#adding metadata
df_cat = df_cat.withColumn("_ingested_at", F.current_timestamp())\
                .withColumn("_soruce_file", F.col("_metadata.file_path"))

#write raw data to the bronze layer(catalog: ecomnmerce , schema:bronze, table: brz_category)
df_cat.write.format("delta")\
        .mode("overwrite")\
        .option("mergeSchema","true")\
        .saveAsTable(f"{catalog_name}.bronze.brz_category")



##Products

In [0]:
##define the schema
product_schema = StructType([
    StructField('product_id', StringType(),False),
    StructField("sku", StringType(),True),
    StructField("category_code", StringType(),True),
    StructField("brand_code", StringType(),True),
    StructField("color", StringType(),True),
    StructField("size",StringType(),True),
    StructField("material",StringType(),True),
    StructField("weight_grams",StringType(),True),
    StructField("length_cm",StringType(),True),
    StructField("width_cm",StringType(),True),
    StructField("height_cm",StringType(),True),
    StructField("rating_count",StringType(),True),
])
#load data using the schema defined
raw_data_path ="/Volumes/ecommerce/source_data/raw/products/*.csv"
df_prod = spark.read.option("header","true").option("delimiter",",").schema(product_schema).csv(raw_data_path)\
            .withColumn("_source_file", F.col("_metadata.file_path"))\
            .withColumn("ingested_at",F.current_timestamp())

#write raw data to the bronze layer(catalog: ecomnmerce , schema:bronze, table:
df_prod.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_products")



##Customers

In [0]:
#define the schema
customer_schema= StructType([
    StructField("customer_id",StringType(),False),
    StructField("phone", StringType(),True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(),True),
    StructField("state",StringType(),True),
    StructField("file_name",StringType(),False),
    StructField("ingest_timestamp",TimestampType(),False)
])

#load data using the schema defined
raw_data_path ="/Volumes/ecommerce/source_data/raw/customers/*.csv"
df_cust = spark.read.option("header","true").option("delimiter",",").schema(customer_schema).csv(raw_data_path)\
                .withColumn("file_name",F.col("_metadata.file_path"))\
                .withColumn("ingest_timestamp",F.current_timestamp())

#write raw data to the bronze layer(catalog: ecomnmerce , schema:bronze, table:
df_cust.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_customers")

##Date

In [0]:

# Define schema for the data file
date_schema = StructType([
    StructField("date", StringType(), True),           # Raw date in string format
    StructField("year", IntegerType(), True),          # Year
    StructField("day_name", StringType(), True),       # Day name (can be mixed case)
    StructField("quarter", IntegerType(), True),       # Quarter
    StructField("week_of_year", IntegerType(), True),  # Week of year (can be negative)
])

# Load data using the schema defined
raw_data_path = f"/Volumes/ecommerce/source_data/raw/date/*.csv" 

df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(date_schema).csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_ingested_at", F.current_timestamp()) \
               .withColumn("_source_file", F.col("_metadata.file_path"))


# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_calendar) 
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_calendar")      